<a href="https://colab.research.google.com/github/sselgrad23/Pandas-Tutorial-For-Beginners/blob/main/pandas_tutorial_lesson_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aggregating Data



In [1]:
import pandas as pd
import numpy as np

In [2]:
bios = pd.read_csv('https://github.com/KeithGalli/complete-pandas-tutorial/raw/refs/heads/master/data/bios.csv')
bios.head()

,athlete_id,name,born_date,born_city,born_region,born_country,NOC,height_cm,weight_kg,died_date
0,1,Jean-François Blanchy,1886-12-12,Bordeaux,Gironde,FRA,France,NaN,NaN,1960-10-02
1,2,Arnaud Boetsch,1969-04-01,Meulan,Yvelines,FRA,France,183.0,76.0,NaN
2,3,Jean Borotra,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA,France,183.0,76.0,1994-07-17
3,4,Jacques Brugnon,1895-05-11,Paris VIIIe,Paris,FRA,France,168.0,64.0,1978-03-20
4,5,Albert Canet,1878-04-17,Wandsworth,England,GBR,France,NaN,NaN,1930-07-25


Let's say we want to find the most popular values in a certain column. We can do this using ```.value_counts()```:



In [3]:
bios['born_city'].value_counts()

,count
born_city,
Budapest,1378
Moskva (Moscow),883
Oslo,708
Stockholm,629
Praha (Prague),600
...,...
Kirovgrad,1
Pereiaslav,1
Podgornyy,1


But what if we want to find out the top 10 most popular birth regions for atheletes in the USA? We just need to do more filtering:

In [4]:
bios[bios['born_country']=='USA']['born_region'].value_counts().head(10)

,count
born_region,
California,1634
New York,990
Illinois,585
Massachusetts,530
Pennsylvania,530
New Jersey,381
Texas,368
Minnesota,365
Ohio,328


Let's go back to the coffee dataset.

In [5]:
coffee = pd.read_csv('https://raw.githubusercontent.com/KeithGalli/complete-pandas-tutorial/refs/heads/master/warmup-data/coffee.csv')
coffee['price'] = np.where(coffee['Coffee Type'] == 'Espresso', 3.99, 5.99)
coffee['revenue'] = coffee['Units Sold'] * coffee['price']
coffee

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,25,3.99,99.75
1,Monday,Latte,15,5.99,89.85
2,Tuesday,Espresso,30,3.99,119.70
3,Tuesday,Latte,20,5.99,119.80
4,Wednesday,Espresso,35,3.99,139.65
5,Wednesday,Latte,25,5.99,149.75
6,Thursday,Espresso,40,3.99,159.60
7,Thursday,Latte,30,5.99,179.70
8,Friday,Espresso,45,3.99,179.55
9,Friday,Latte,35,5.99,209.65


A useful function for aggregating data is `.groupby()`.

Imagine you have a list of sales data, and you want to know the total sales for each product, or the average rating for each movie genre. Instead of manually sifting through all the data, `.groupby()` lets you automatically organize and summarize your data by categories. This makes it super easy to get insights from your data quickly!

Let's find out the total number of Espressos and Lattes sold:

In [6]:
coffee.groupby(['Coffee Type'])['Units Sold'].sum()

,Units Sold
Coffee Type,
Espresso,265
Latte,195


Or perhaps we want to know the average number of Espressos and Lattes sold per day:

In [7]:
coffee.groupby(['Coffee Type'])['Units Sold'].mean()

,Units Sold
Coffee Type,
Espresso,37.857143
Latte,27.857143


`.agg` is also useful.

While `groupby()` helps you group data, `.agg()` (short for 'aggregate') is what lets you *do* things with those groups. It's powerful because it allows you to apply different operations to different columns within each group, all at once! For example, you might want to find the total sales for one column (`sum`), but the average price for another (`mean`) for each coffee type. `.agg()` makes this very clean and efficient, giving you a customized summary of your grouped data.

Let's use it to find the average price and the sum of units sold for each type of coffee:

In [8]:
coffee.groupby(['Coffee Type']).agg({'Units Sold': 'sum', 'price': 'mean'})

,Units Sold,price
Coffee Type,,
Espresso,265,3.99
Latte,195,5.99


We can also group by multiple different columns:

In [9]:
coffee.groupby(['Coffee Type', 'Day']).agg({'Units Sold': 'sum', 'price': 'mean'})

Units Sold  price
Coffee Type Day                         
Espresso    Friday             45   3.99
            Monday             25   3.99
            Saturday           45   3.99
            Sunday             45   3.99
            Thursday           40   3.99
            Tuesday            30   3.99
            Wednesday          35   3.99
Latte       Friday             35   5.99
            Monday             15   5.99
            Saturday           35   5.99
            Sunday             35   5.99
            Thursday           30   5.99
            Tuesday            20   5.99
            Wednesday          25   5.99

`.pivot()` is a fantastic way to reshape your data. Imagine you have a table where different types of coffee sales are all listed in one column, and you want to see each coffee type as its own column, with days as rows. `.pivot()` helps you do exactly that! It takes unique values from one of your columns and turns them into new columns, making your data much easier to read and analyze, especially when you want to compare values across different categories side-by-side. It's like reorganizing a spreadsheet to get a different view of the same information.

In [10]:
pivot = coffee.pivot(columns='Coffee Type', index='Day', values='revenue')
pivot

Coffee Type,Espresso,Latte
Day,,
Friday,179.55,209.65
Monday,99.75,89.85
Saturday,179.55,209.65
Sunday,179.55,209.65
Thursday,159.60,179.70
Tuesday,119.70,119.80
Wednesday,139.65,149.75


Now we can grab certain values much more easily. We can find out the revenue generated for lattes on Mondays by doing the following with our new pivot table:

In [11]:
pivot.loc['Monday','Latte']

np.float64(89.85000000000001)

Or we can find the total revenue for each coffee type by doing:

In [12]:
pivot.sum()

,0
Coffee Type,
Espresso,1057.35
Latte,1168.05


By changing the axis, we can find the total revenue on different days of the week:

In [13]:
pivot.sum(axis=1)

,0
Day,
Friday,389.2
Monday,189.6
Saturday,389.2
Sunday,389.2
Thursday,339.3
Tuesday,239.5
Wednesday,289.4


Back to our bios table. Let's find the most popular year for athletes to be born.

We convert `born_date` to a datetime first so we can extract just the year with `.dt.year`. Then we group by year and count how many athletes were born in each one.
`.reset_index()` is needed here because after a `groupby().count()`, pandas turns the grouped column (`born_date`) into the index rather than a regular column. Without it, `born_date` would be the index and you couldn't sort or work with it like a normal column. `.reset_index()` promotes it back to a regular column and gives the DataFrame a clean 0, 1, 2... index instead.
Finally, `.sort_values('name', ascending=False)` sorts by the count column so the most popular birth years appear at the top.

In [14]:
bios['born_date'] = pd.to_datetime(bios['born_date'])
bios.groupby(bios['born_date'].dt.year)['name'].count().reset_index().sort_values('name', ascending=False)

,born_date,name
139,1972.0,2231
152,1985.0,2227
140,1973.0,2216
138,1971.0,2205
137,1970.0,2174
...,...,...
4,1837.0,1
2,1833.0,1
6,1839.0,1
12,1845.0,1


We could even group by the birth month and year:

In [15]:
bios['month_born'] = bios['born_date'].dt.month
bios['year_born'] = bios['born_date'].dt.year
bios.groupby(['year_born', 'month_born'])['name'].count().reset_index().sort_values('name', ascending=False)

,year_born,month_born,name
1437,1970.0,1.0,239
1461,1972.0,1.0,229
1497,1975.0,1.0,227
1629,1986.0,1.0,227
1617,1985.0,1.0,225
...,...,...,...
1877,2006.0,12.0,1
1871,2006.0,3.0,1
20,1846.0,7.0,1
21,1846.0,8.0,1
